In [2]:
#Importar bibiotecas
import io
from pathlib import Path
import pandas as pd

#definir rutas de directorios
RAW_DIR = Path.cwd().parent.parent / 'data' / 'data' / 'raw'
PROCESSED_DIR = Path.cwd().parent.parent / 'data' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Leer el archivo en bytes, le quita el BOM si lo tiene, y lo decodifica como latin-1 para que pandas lo lea sin errores de codificación.
def leer_texto_limpio(ruta):
    raw = ruta.read_bytes()
    if raw.startswith(b'\xef\xbb\xbf'):
        # tiene BOM -> el archivo SÍ está en UTF-8 de verdad; utf-8-sig quita el BOM solo
        return io.StringIO(raw.decode('utf-8-sig'))
    # sin BOM -> estos exports del INE vienen en latin-1
    return io.StringIO(raw.decode('latin-1'))

#Para cargar los datos de población de Gipuzkoa desde un archivo CSV, 
def cargar_poblacion_gipuzkoa(nombre_archivo, nombre_variable, nombre_salida):
    #leer csv de forma correcta
    df = pd.read_csv(
        leer_texto_limpio(RAW_DIR / nombre_archivo),
        sep='\t',
        thousands='.',
        decimal=',',
    )
    #Cambiar nombres de columnas y eliminar columna innecesariaa
    df = df.rename(columns={
        'Municipios': 'municipio_raw',
        'Secciones': 'seccion_raw',
        'Sexo': 'sexo',
        nombre_variable: 'variable',
        'Periodo': 'anio',
        'Total': 'poblacion',
    })
    df = df.drop(columns=['Provincias'])

   # la columna que sobra después de renombrar las fijas es la variable propia de esta tabla
    columnas_fijas = {'municipio_raw', 'seccion_raw', 'sexo', 'anio', 'poblacion'}
    columna_variable = [c for c in df.columns if c not in columnas_fijas][0]
    df = df.rename(columns={columna_variable: 'variable'})

    # Clasifica cada fila como provincia, municipio o sección según qué columnas de geografía vienen vacías, y guarda el resultado en una nueva columna 'nivel'
    def nivel(row):
        if pd.isna(row['municipio_raw']):
            return 'provincia'
        elif pd.isna(row['seccion_raw']):
            return 'municipio'
        return 'seccion'

    df['nivel'] = df.apply(nivel, axis=1)

    #Divide la columna 'municipio_raw' en dos nuevas columnas: 'municipio_codigo' y 'municipio_nombre', usando una expresión regular para extraer el código y el nombre del municipio.
    muni_split = df['municipio_raw'].str.extract(r'^(\d+)\s+(.*)$')
    df['municipio_codigo'] = muni_split[0]
    df['municipio_nombre'] = muni_split[1]

    df.to_csv(PROCESSED_DIR / nombre_salida, index=False)
    return df

In [4]:
#Cargar los 4 csv de población de Gipuzkoa, cada uno con su variable propia, y guardarlos en archivos procesados
df1 = cargar_poblacion_gipuzkoa('Población1.csv', 'Nacionalidad', 'poblacion1_nacionalidad.csv')
df2 = cargar_poblacion_gipuzkoa('Población2.csv', 'Lugar de nacimiento', 'poblacion2_pais_nacimiento_simple.csv')
df3 = cargar_poblacion_gipuzkoa('Población3.csv', 'Relación entre lugar de nacimiento y lugar de residencia', 'poblacion3_relacion_nacimiento_residencia.csv')
df4 = cargar_poblacion_gipuzkoa('Población4.csv', 'País de nacimiento', 'poblacion4_pais_nacimiento_detalle.csv')

# Etiquetar cada tabla antes de unirlas, para saber después de dónde viene cada fila
df1['tabla'] = 'nacionalidad'
df2['tabla'] = 'pais_nacimiento_simple'
df3['tabla'] = 'relacion_nacimiento_residencia'
df4['tabla'] = 'pais_nacimiento_detalle'

# pd.concat apila los 4 DataFrames uno debajo de otro (funciona porque tienen las mismas columnas)
poblacion = pd.concat([df1, df2, df3, df4], ignore_index=True)

poblacion.to_csv(PROCESSED_DIR / 'poblacion_gipuzkoa_combinada.csv', index=False)
poblacion.head()

,municipio_raw,seccion_raw,sexo,variable,anio,poblacion,nivel,municipio_codigo,municipio_nombre,tabla
0,NaN,NaN,Hombres,Española,2025,320128.0,provincia,NaN,NaN,nacionalidad
1,NaN,NaN,Hombres,Española,2024,320800.0,provincia,NaN,NaN,nacionalidad
2,NaN,NaN,Hombres,Española,2023,321629.0,provincia,NaN,NaN,nacionalidad
3,NaN,NaN,Hombres,Española,2022,323058.0,provincia,NaN,NaN,nacionalidad
4,NaN,NaN,Hombres,Española,2021,324345.0,provincia,NaN,NaN,nacionalidad


In [5]:
#REVISIÓN INICIAL DE LA TABLA COMBINADA
#Resumen general de la tabla combinada
poblacion.info()

#Contador de valores nulos y duplicados
print("Valores nulos:")
print(poblacion.isna().sum())
print("Valores duplicados:")
print(poblacion.duplicated().sum())

# valores únicos de sexo
print(poblacion['sexo'].unique())

# valores únicos de 'variable', tabla por tabla (cada una tiene categorías distintas)
for t in poblacion['tabla'].unique():
    print(t, '->', poblacion[poblacion['tabla'] == t]['variable'].unique())
    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 283465 entries, 0 to 283464
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   municipio_raw     283020 non-null  object 
 1   seccion_raw       243860 non-null  object 
 2   sexo              283465 non-null  object 
 3   variable          283465 non-null  object 
 4   anio              283465 non-null  int64  
 5   poblacion         282219 non-null  float64
 6   nivel             283465 non-null  object 
 7   municipio_codigo  283020 non-null  object 
 8   municipio_nombre  283020 non-null  object 
 9   tabla             283465 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 21.6+ MB
Valores nulos:
municipio_raw         445
seccion_raw         39605
sexo                    0
variable                0
anio                    0
poblacion            1246
nivel                   0
municipio_codigo      445
municipio_nombre      445
t

In [6]:
#checar de donde vienen los valores nulos
nulos = poblacion[poblacion['poblacion'].isna()]
print(nulos['nivel'].value_counts())
print(nulos['tabla'].value_counts())

print(poblacion.isna().groupby(poblacion['nivel']).sum())

nivel
seccion    1246
Name: count, dtype: int64
tabla
pais_nacimiento_detalle           882
relacion_nacimiento_residencia    252
nacionalidad                       56
pais_nacimiento_simple             56
Name: count, dtype: int64
           municipio_raw  seccion_raw  sexo  variable  anio  poblacion  nivel  \
nivel                                                                           
municipio              0        39160     0         0     0          0      0   
provincia            445          445     0         0     0          0      0   
seccion                0            0     0         0     0       1246      0   

           municipio_codigo  municipio_nombre  tabla  
nivel                                                 
municipio                 0                 0      0  
provincia               445               445      0  
seccion                   0                 0      0  


In [7]:
#Falta de dato por secreto estadístico 
poblacion['dato_oculto'] = poblacion['poblacion'].isna()
poblacion = poblacion.drop(columns=['dato_oculto'])

poblacion.head()

,municipio_raw,seccion_raw,sexo,variable,anio,poblacion,nivel,municipio_codigo,municipio_nombre,tabla
0,NaN,NaN,Hombres,Española,2025,320128.0,provincia,NaN,NaN,nacionalidad
1,NaN,NaN,Hombres,Española,2024,320800.0,provincia,NaN,NaN,nacionalidad
2,NaN,NaN,Hombres,Española,2023,321629.0,provincia,NaN,NaN,nacionalidad
3,NaN,NaN,Hombres,Española,2022,323058.0,provincia,NaN,NaN,nacionalidad
4,NaN,NaN,Hombres,Española,2021,324345.0,provincia,NaN,NaN,nacionalidad


In [8]:
#Eliminar espacions en blanco al inicio y final de los valores de las columnas de tipo texto
columnas_texto = poblacion.select_dtypes(include='object').columns
for col in columnas_texto:
    poblacion[col] = poblacion[col].str.strip()

#Poner nombres a las filas de provincia
es_prov = poblacion['nivel'] == 'provincia'
poblacion.loc[es_prov, 'municipio_nombre'] = 'Gipuzkoa'
poblacion.loc[es_prov, 'municipio_codigo'] = '20'

# Convertir población a números enteros, reemplazando los valores nulos por 0
poblacion['poblacion'] = poblacion['poblacion'].fillna(0).astype(int)

#Verificar que las 4 tablas de población sumen correctamente a nivel de provincia y municipio
m = poblacion[poblacion['nivel'].isin(['provincia', 'municipio'])]
prov = m[(m['nivel'] == 'provincia') & (m['sexo'] == 'Total') & (m['variable'] != 'Total')]
print(prov.pivot_table(index='anio', columns='tabla', values='poblacion', aggfunc='sum'))

tabla  pais_nacimiento_detalle  relacion_nacimiento_residencia
anio                                                          
2021                    725508                          725508
2022                    723241                          723241
2023                    726712                          726712
2024                    729722                          729722
2025                    733149                          733149


In [9]:
#Organizar la columnna total de sexo y variable para que sea consistente en todas las tablas
print(poblacion['sexo'].unique())

for t in poblacion['tabla'].unique():
    cats = poblacion.loc[poblacion['tabla'] == t, 'variable'].unique()
    print(t, '->', cats)

#Verificar que tablas tienen la columna 'Total' en sexo y variable, y que tablas no la tienen
print(poblacion.groupby('tabla')['sexo'].unique())

['Hombres' 'Mujeres' 'Total']
nacionalidad -> ['Española' 'Extranjera']
pais_nacimiento_simple -> ['España' 'Extranjero']
relacion_nacimiento_residencia -> ['Total' 'Mismo municipio al de residencia'
 'Distinto municipio de la misma provincia'
 'Distinta provincia de la misma comunidad' 'Distinta comunidad'
 'Nacido en el extranjero o en antiguos territorios españoles']
pais_nacimiento_detalle -> ['Total' 'España' 'Francia' 'Reino Unido' 'Rumanía' 'Ucrania'
 'Otros países de Europa' 'Marruecos' 'Otros países de África' 'Cuba'
 'República Dominicana' 'Argentina' 'Bolivia' 'Colombia' 'Ecuador' 'Perú'
 'Venezuela' 'Otros países de América' 'China' 'Otros países de Asia'
 'Oceanía']
tabla
nacionalidad                             [Hombres, Mujeres]
pais_nacimiento_detalle           [Total, Hombres, Mujeres]
pais_nacimiento_simple                   [Hombres, Mujeres]
relacion_nacimiento_residencia    [Total, Hombres, Mujeres]
Name: sexo, dtype: object


In [10]:
# Quitar todas las filas "Total" (se pueden recalcular sumando)
poblacion_limpia = poblacion[(poblacion['sexo'] != 'Total') &
                             (poblacion['variable'] != 'Total')].copy()

print(poblacion_limpia['sexo'].unique())   # deverificar que solo salga Hombres y Mujeres

poblacion_limpia.head()

['Hombres' 'Mujeres']


,municipio_raw,seccion_raw,sexo,variable,anio,poblacion,nivel,municipio_codigo,municipio_nombre,tabla
0,NaN,NaN,Hombres,Española,2025,320128,provincia,20,Gipuzkoa,nacionalidad
1,NaN,NaN,Hombres,Española,2024,320800,provincia,20,Gipuzkoa,nacionalidad
2,NaN,NaN,Hombres,Española,2023,321629,provincia,20,Gipuzkoa,nacionalidad
3,NaN,NaN,Hombres,Española,2022,323058,provincia,20,Gipuzkoa,nacionalidad
4,NaN,NaN,Hombres,Española,2021,324345,provincia,20,Gipuzkoa,nacionalidad


In [11]:
p = poblacion_limpia[poblacion_limpia['nivel'] == 'provincia']
print(p.pivot_table(index='anio', columns='tabla', values='poblacion', aggfunc='sum'))

tabla  nacionalidad  pais_nacimiento_detalle  pais_nacimiento_simple  \
anio                                                                   
2021         725508                   725508                  725508   
2022         723241                   723241                  723241   
2023         726712                   726712                  726712   
2024         729722                   729722                  729722   
2025         733149                   733149                  733149   

tabla  relacion_nacimiento_residencia  
anio                                   
2021                           725508  
2022                           723241  
2023                           726712  
2024                           729722  
2025                           733149  


In [12]:
# Limpiar columnas innecesarias
poblacion_limpia = poblacion_limpia.drop(columns=['municipio_raw'])
poblacion_limpia = poblacion_limpia.rename(columns={'seccion_raw': 'seccion'})

In [13]:
#Verificar que no hay valores nulos en la tabla limpia y si hay de qué nivel geográfico son
print(poblacion_limpia.isna().groupby(poblacion_limpia['nivel']).sum())
poblacion_limpia.head()

           seccion  sexo  variable  anio  poblacion  nivel  municipio_codigo  \
nivel                                                                          
municipio    25520     0         0     0          0      0                 0   
provincia      290     0         0     0          0      0                 0   
seccion          0     0         0     0          0      0                 0   

           municipio_nombre  tabla  
nivel                               
municipio                 0      0  
provincia                 0      0  
seccion                   0      0  


,seccion,sexo,variable,anio,poblacion,nivel,municipio_codigo,municipio_nombre,tabla
0,NaN,Hombres,Española,2025,320128,provincia,20,Gipuzkoa,nacionalidad
1,NaN,Hombres,Española,2024,320800,provincia,20,Gipuzkoa,nacionalidad
2,NaN,Hombres,Española,2023,321629,provincia,20,Gipuzkoa,nacionalidad
3,NaN,Hombres,Española,2022,323058,provincia,20,Gipuzkoa,nacionalidad
4,NaN,Hombres,Española,2021,324345,provincia,20,Gipuzkoa,nacionalidad


In [14]:
#guardar tabla limpia en archivo CSV
from pathlib import Path
ruta = Path.cwd().parent.parent / 'data' / 'data' / 'processed' / 'poblacion_limpia.csv'
poblacion_limpia.to_csv(ruta, index=False, encoding='utf-8')
print('Guardado en', ruta)

Guardado en /Users/anamariausabiagaalvarez/Movilidad_Gipuzcoa/data/data/processed/poblacion_limpia.csv


In [15]:
#estadísticas descriptivas de la población por nivel geográfico
df = poblacion_limpia
#Españoles y extranjeros por año:
df[(df['nivel'] == 'provincia') & (df['tabla'] == 'nacionalidad')] \
    .pivot_table(index='anio', columns='variable', values='poblacion', aggfunc='sum')

variable,Española,Extranjera
anio,,
2021,665445,60063
2022,663160,60081
2023,660601,66111
2024,659271,70451
2025,658533,74616


In [16]:
#Población total de Gipuzkoa por año:
df = poblacion_limpia
df[(df['nivel'] == 'provincia') & (df['tabla'] == 'nacionalidad')] \
    .groupby('anio')['poblacion'].sum()

anio
2021    725508
2022    723241
2023    726712
2024    729722
2025    733149
Name: poblacion, dtype: int64

In [17]:
#Población de cada municipio por año:
df[(df['nivel'] == 'municipio') & (df['tabla'] == 'nacionalidad')] \
    .pivot_table(index='municipio_nombre', columns='anio', values='poblacion', aggfunc='sum')

anio,2021,2022,2023,2024,2025
municipio_nombre,,,,,
Abaltzisketa,312,308,326,316,320
Aduna,478,486,505,498,500
Aia,2053,2090,2135,2139,2148
Aizarnazabal,793,773,771,780,822
Albiztur,315,316,325,324,328
...,...,...,...,...,...
Zerain,282,276,282,281,285
Zestoa,3757,3738,3777,3842,3932
Zizurkil,3002,2988,2967,2966,2984
